# CareerMatch – Top-3 Career Role Recommendation Using KNN

This notebook contains the final mini-project workflow using the original 1,500-row dataset.

CareerMatch is treated as a **career recommendation system** rather than a single exact-career classifier.  
For each student profile, the model returns the **Top 3 recommended career roles** using K-Nearest Neighbors (KNN).

The main evaluation metric used in this notebook is **Top-3 Recommendation Coverage**:
> A prediction is counted as correct when the actual career appears anywhere among the three recommended careers.


## 1. Dataset Loading and Initial Exploration


In [ ]:
import pandas as pd

df = pd.read_csv("../data/perfectly_realistic_career_guidance_dataset_1500.csv")

df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df["Recommended_Career"].value_counts()

In [ ]:
df.nunique()

In [ ]:
X = df[
    [
        "Education_Level",
        "Specialization",
        "Skills",
        "Interests"
    ]
]

y = df["Recommended_Career"]

print("Input shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
for col in ["Education_Level", "Specialization", "Interests"]:
    print(f"\n{col}:")
    print(df[col].unique())

In [ ]:
print(df["Skills"].unique())

In [ ]:
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
import pandas as pd

In [ ]:
# -------------------------------
# 1. One-Hot Encode categorical columns
# -------------------------------

categorical_columns = [
    "Education_Level",
    "Specialization",
    "Interests"
]

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

categorical_encoded = encoder.fit_transform(
    df[categorical_columns]
)

categorical_encoded.shape

In [ ]:
# Convert comma-separated skill strings into lists

skills_list = df["Skills"].apply(
    lambda x: [skill.strip() for skill in x.split(",")]
)

skills_list.head()

In [ ]:
skill_encoder = MultiLabelBinarizer()

skills_encoded = skill_encoder.fit_transform(skills_list)

skills_encoded.shape

In [ ]:
skill_encoder.classes_

In [ ]:
# Get proper column names for one-hot encoded categorical features
categorical_feature_names = encoder.get_feature_names_out(categorical_columns)

categorical_df = pd.DataFrame(
    categorical_encoded,
    columns=categorical_feature_names,
    index=df.index
)

# Proper column names for skill features
skills_df = pd.DataFrame(
    skills_encoded,
    columns=[f"Skill_{skill}" for skill in skill_encoder.classes_],
    index=df.index
)

# Combine both
X_encoded = pd.concat(
    [categorical_df, skills_df],
    axis=1
)

print("Encoded shape:", X_encoded.shape)
print("Duplicate columns:", X_encoded.columns.duplicated().sum())

## 2. Duplicate-Profile Analysis and Group-Aware Split

The dataset contains repeated student profiles. To avoid data leakage, identical profiles are kept entirely in either training or testing data.

This makes the final evaluation more realistic because the model is tested on profile groups it has not seen during training.


In [ ]:
feature_columns = [
    "Education_Level",
    "Specialization",
    "Skills",
    "Interests"
]

duplicate_profiles = df.duplicated(
    subset=feature_columns,
    keep=False
).sum()

print("Rows with duplicate profiles:", duplicate_profiles)

In [ ]:
unique_profiles = df[feature_columns].drop_duplicates().shape[0]

print("Total rows:", len(df))
print("Unique feature profiles:", unique_profiles)

In [ ]:
career_counts_per_profile = (
    df.groupby(feature_columns)["Recommended_Career"]
      .nunique()
)

print(
    "Profiles linked to multiple careers:",
    (career_counts_per_profile > 1).sum()
)

In [ ]:
feature_columns = [
    "Education_Level",
    "Specialization",
    "Skills",
    "Interests"
]

profile_groups = (
    df[feature_columns]
    .astype(str)
    .agg(" | ".join, axis=1)
)

print("Unique profile groups:", profile_groups.nunique())

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X_encoded, y, groups=profile_groups)
)

X_train = X_encoded.iloc[train_idx]
X_test = X_encoded.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
train_profiles = set(profile_groups.iloc[train_idx])
test_profiles = set(profile_groups.iloc[test_idx])

overlap = train_profiles.intersection(test_profiles)

print("Profiles present in both train and test:", len(overlap))

## 3. Top-3 Based KNN Validation and K Selection

Because CareerMatch gives **three career recommendations**, K is selected using **Top-3 recommendation coverage on the validation set**, not Top-1 accuracy.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

train_groups = profile_groups.iloc[train_idx]

gss_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

subtrain_pos, val_pos = next(
    gss_val.split(
        X_train,
        y_train,
        groups=train_groups
    )
)

X_subtrain = X_train.iloc[subtrain_pos]
X_val = X_train.iloc[val_pos]

y_subtrain = y_train.iloc[subtrain_pos]
y_val = y_train.iloc[val_pos]

print("Sub-train shape:", X_subtrain.shape)
print("Validation shape:", X_val.shape)


In [ ]:
def top3_coverage(model, X_data, y_true):
    probabilities = model.predict_proba(X_data)

    top3_indices = np.argsort(
        probabilities,
        axis=1
    )[:, -3:][:, ::-1]

    top3_predictions = model.classes_[top3_indices]

    correct = sum(
        actual in recommendations
        for actual, recommendations
        in zip(y_true, top3_predictions)
    )

    return correct / len(y_true)


In [ ]:
top3_validation_results = []

for k in range(1, 31):
    model = KNeighborsClassifier(
        n_neighbors=k,
        weights="uniform",
        p=2
    )

    model.fit(X_subtrain, y_subtrain)

    coverage = top3_coverage(
        model,
        X_val,
        y_val
    )

    top3_validation_results.append((coverage, k))

top3_validation_results.sort(reverse=True)

print("Best Top-3 validation results:")
for coverage, k in top3_validation_results[:10]:
    print(
        f"K = {k} | "
        f"Top-3 Coverage = {coverage * 100:.2f}%"
    )


In [ ]:
best_top3_coverage, best_k = top3_validation_results[0]

print("Selected K:", best_k)
print(
    f"Best Validation Top-3 Coverage: "
    f"{best_top3_coverage * 100:.2f}%"
)


In [ ]:
import matplotlib.pyplot as plt

k_values = sorted([k for _, k in top3_validation_results])

coverage_by_k = {
    k: coverage
    for coverage, k in top3_validation_results
}

coverage_values = [
    coverage_by_k[k]
    for k in k_values
]

plt.figure(figsize=(10, 5))
plt.plot(k_values, coverage_values, marker="o")
plt.xlabel("Number of Neighbors (K)")
plt.ylabel("Top-3 Recommendation Coverage")
plt.title("Top-3 Recommendation Coverage for Different K Values")
plt.show()


## 4. Final Top-3 KNN Recommendation Model

The final model is trained on the full group-aware training set using the K selected from validation.

For this dataset and split, the validation procedure selects **K = 24**.


In [ ]:
final_knn = KNeighborsClassifier(
    n_neighbors=24,
    weights="uniform",
    p=2
)

final_knn.fit(X_train, y_train)

print("Final KNN model trained successfully.")


## 5. Final Top-3 Recommendation Evaluation


In [ ]:
final_top3_coverage = top3_coverage(
    final_knn,
    X_test,
    y_test
)

print(
    f"Final Top-3 Recommendation Coverage: "
    f"{final_top3_coverage * 100:.2f}%"
)


### Interpretation

The final metric answers:

> **For how many test students did the actual career appear among CareerMatch's three recommendations?**

This is more suitable for career guidance than forcing the model to choose only one exact career, because similar skill and interest profiles can reasonably match multiple career roles.


## 6. Top-3 Career Recommendation Function


In [ ]:
def get_top3_recommendations(model, X_input):
    probabilities = model.predict_proba(X_input)[0]

    top3_indices = np.argsort(
        probabilities
    )[-3:][::-1]

    top3_careers = model.classes_[top3_indices]
    top3_scores = probabilities[top3_indices]

    return list(
        zip(top3_careers, top3_scores)
    )


In [ ]:
def prepare_student_input(
    education,
    specialization,
    interest,
    skills
):
    # Encode Education Level, Specialization and Interest
    categorical_input = pd.DataFrame({
        "Education_Level": [education],
        "Specialization": [specialization],
        "Interests": [interest]
    })

    categorical_encoded_input = encoder.transform(
        categorical_input
    )

    categorical_input_df = pd.DataFrame(
        categorical_encoded_input,
        columns=encoder.get_feature_names_out(
            categorical_columns
        )
    )

    # Encode multiple selected skills
    skills_encoded_input = skill_encoder.transform(
        [skills]
    )

    skills_input_df = pd.DataFrame(
        skills_encoded_input,
        columns=[
            f"Skill_{skill}"
            for skill in skill_encoder.classes_
        ]
    )

    # Combine all encoded features
    student_encoded = pd.concat(
        [
            categorical_input_df,
            skills_input_df
        ],
        axis=1
    )

    # Match training feature order
    student_encoded = student_encoded.reindex(
        columns=X_encoded.columns,
        fill_value=0
    )

    return student_encoded


## 7. Example Career Recommendation


In [ ]:
student = prepare_student_input(
    education="BCA",
    specialization="Computer Applications",
    interest="Technology",
    skills=[
        "Python",
        "Web Development",
        "Databases"
    ]
)

recommendations = get_top3_recommendations(
    final_knn,
    student
)

print("Top 3 Career Recommendations:\n")

for i, (career, score) in enumerate(
    recommendations,
    start=1
):
    print(
        f"{i}. {career} "
        f"- Recommendation Score: {score * 100:.2f}%"
    )


### About the Recommendation Score

The recommendation score is the share of the K nearest neighbours voting for each career.

It should be interpreted as a **KNN recommendation score**, not as a guarantee of career success or an absolute probability.


## 8. Optional Experiments

These experiments compare alternative KNN distance metrics and feature-weighting strategies.

They are kept only as supporting analysis and **do not affect the final Top-3 model above**.


In [ ]:
# Hamming distance experiment
hamming_results = []

for k in range(1, 31):
    model = KNeighborsClassifier(
        n_neighbors=k,
        metric="hamming",
        algorithm="brute"
    )

    model.fit(X_subtrain, y_subtrain)

    coverage = top3_coverage(
        model,
        X_val,
        y_val
    )

    hamming_results.append((coverage, k))

hamming_results.sort(reverse=True)

print("Best Hamming Top-3 results:")
for coverage, k in hamming_results[:10]:
    print(
        f"Top-3 Coverage: {coverage * 100:.2f}% | K: {k}"
    )


In [ ]:
# Cosine distance experiment
cosine_results = []

for k in range(1, 31):
    model = KNeighborsClassifier(
        n_neighbors=k,
        metric="cosine",
        algorithm="brute"
    )

    model.fit(X_subtrain, y_subtrain)

    coverage = top3_coverage(
        model,
        X_val,
        y_val
    )

    cosine_results.append((coverage, k))

cosine_results.sort(reverse=True)

print("Best Cosine Top-3 results:")
for coverage, k in cosine_results[:10]:
    print(
        f"Top-3 Coverage: {coverage * 100:.2f}% | K: {k}"
    )


In [ ]:
# Feature weighting experiment
X_weighted = X_encoded.copy()

for col in X_weighted.columns:
    if col.startswith("Skill_"):
        X_weighted[col] *= 2.5
    elif col.startswith("Specialization_"):
        X_weighted[col] *= 2.0
    elif col.startswith("Interests_"):
        X_weighted[col] *= 1.5

print("Weighted feature matrix created.")


In [ ]:
train_idx_w, test_idx_w = next(
    gss.split(
        X_weighted,
        y,
        groups=profile_groups
    )
)

X_train_w = X_weighted.iloc[train_idx_w]
X_test_w = X_weighted.iloc[test_idx_w]

y_train_w = y.iloc[train_idx_w]
y_test_w = y.iloc[test_idx_w]

train_groups_w = profile_groups.iloc[train_idx_w]

subtrain_pos_w, val_pos_w = next(
    gss_val.split(
        X_train_w,
        y_train_w,
        groups=train_groups_w
    )
)

X_subtrain_w = X_train_w.iloc[subtrain_pos_w]
X_val_w = X_train_w.iloc[val_pos_w]

y_subtrain_w = y_train_w.iloc[subtrain_pos_w]
y_val_w = y_train_w.iloc[val_pos_w]


In [ ]:
weighted_results = []

for k in range(1, 31):
    model = KNeighborsClassifier(
        n_neighbors=k,
        weights="uniform",
        p=2
    )

    model.fit(
        X_subtrain_w,
        y_subtrain_w
    )

    coverage = top3_coverage(
        model,
        X_val_w,
        y_val_w
    )

    weighted_results.append(
        (coverage, k)
    )

weighted_results.sort(reverse=True)

print("Best Weighted-Feature Top-3 results:")
for coverage, k in weighted_results[:10]:
    print(
        f"Top-3 Coverage: {coverage * 100:.2f}% | K: {k}"
    )
